# 01 - Build DPO preference datasets from AutoBench scores

Turns the raw AutoBench export (answers + peer-assessment scores) into DPO-ready
`(prompt, chosen, rejected)` datasets.

**Pipeline**
1. Load the CSV and repair mojibake with `ftfy`
2. Filter answers by length
3. Extract preference pairs (`best_worst` or `pairwise`) with a score-gap filter (`min_delta`), a Jaccard similarity filter and a per-prompt cap
4. Split **by prompt** (90/10, seed 42) to avoid train/test leakage

> The AutoBench data is private and is **not** distributed with this repository.
> Set `RAW_CSV` below to your own export. The CSV must contain the columns
> `question`, `answer`, `average_rank_os` (mean 1-5 score from the open-source judges) and `model_name`.
> A command-line version of the same logic lives in `src/build_dataset.py`.

> 🔒 **Data availability.** The preference datasets used in this work (and the AutoBench export they are derived from) were shared privately by the AutoBench authors and are **not included in this repository**. All paths in this notebook are placeholders (`<your-...>`): point them at your own data with the same schema (`prompt`, `chosen`, `rejected`). Cell outputs have been cleared so that no private data is exposed.

In [ ]:
# ============================================================
# CONFIG - edit the paths for your environment
# ============================================================
# NOTE: private data - not distributed with this repo. Replace the placeholder path below.
RAW_CSV = "/kaggle/input/<your-autobench-dataset>/<autobench_export>.csv"
OUTPUT_DIR = "/kaggle/working"

## 1. Load data and fix text encoding (ftfy)

In [ ]:
# ftfy recognises (even multi-level) mojibake patterns and repairs them. A manual
# cp1252 re-encoding approach failed on ~40% of the rows, because a single
# foreign character made the whole row fail to encode.
!pip install ftfy --quiet

import ftfy
import pandas as pd

df_raw = pd.read_csv(RAW_CSV, encoding="utf-8")
print(f"Rows loaded: {len(df_raw)}\n")

MOJIBAKE_MARKER = "\u00e2\u20ac"  # typical start of a UTF-8 -> cp1252 mojibake sequence ("â€")
n_before = df_raw["answer"].astype(str).str.contains(MOJIBAKE_MARKER, regex=False).sum()
print(f"Rows with visible mojibake BEFORE the fix: {n_before} / {len(df_raw)}")

df = df_raw.copy()
df["question"] = df["question"].astype(str).map(ftfy.fix_text)
df["answer"] = df["answer"].astype(str).map(ftfy.fix_text)

n_after = df["answer"].str.contains(MOJIBAKE_MARKER, regex=False).sum()
print(f"Rows with visible mojibake AFTER the fix : {n_after} / {len(df)}")
if n_after:
    print(f"WARNING: {n_after} rows still contain mojibake - inspect them separately.")

## 2. Helper functions

In [ ]:
from itertools import combinations


def jaccard_similarity(text_a, text_b):
    """Jaccard similarity over the sets of lower-cased whitespace tokens."""
    set_a = set(str(text_a).lower().split())
    set_b = set(str(text_b).lower().split())
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def format_dpo_example(question, chosen_answer, rejected_answer, conversational_format=True):
    if conversational_format:
        prompt_data = [{"role": "user", "content": question}]
        chosen_data = [{"role": "assistant", "content": chosen_answer}]
        rejected_data = [{"role": "assistant", "content": rejected_answer}]
    else:
        prompt_data = (f"<|im_start|>user\n{question}<|im_end|>\n"
                       f"<|im_start|>assistant\n")
        chosen_data = f"{chosen_answer}<|im_end|>"
        rejected_data = f"{rejected_answer}<|im_end|>"
    return prompt_data, chosen_data, rejected_data


def filter_answer_length(df, min_words=3, max_words=2000, col_answer="answer"):
    """Drop empty or absurdly long answers before forming pairs (and report how many)."""
    df = df.copy()
    n_words = df[col_answer].astype(str).str.split().map(len)
    before = len(df)
    df = df[(n_words >= min_words) & (n_words <= max_words)]
    print(f"Answer-length filter: dropped {before - len(df)} / {before} rows "
          f"(outside [{min_words}, {max_words}] words)")
    return df

## 3. Dataset builder (`best_worst` / `pairwise`)

In [ ]:
def build_dpo_dataset(
    df,
    output_parquet_path,
    strategy="best_worst",          # "pairwise" or "best_worst"
    min_delta=0.3,
    similarity_threshold=0.85,
    max_pairs_per_prompt=None,
    min_answer_words=3,
    max_answer_words=2000,
    conversational_format=True,
    test_size=0.1,                  # fraction of unique prompts held out for testing
    random_state=42,                # seed for a deterministic, reproducible split
):
    df = df.dropna(subset=["question", "answer", "average_rank_os"]).copy()
    df = filter_answer_length(df, min_words=min_answer_words, max_words=max_answer_words)

    answers_per_prompt = df.groupby("question").size()
    print(f"\nUnique prompts: {len(answers_per_prompt)}")
    print(f"Mean answers per prompt: {answers_per_prompt.mean():.2f}")
    print(f"Strategy: {strategy}\n")

    dpo_list = []
    n_skip_few = n_skip_delta = n_skip_sim = 0   # discard-reason counters

    for question, group in df.groupby("question"):
        if len(group) < 2:
            n_skip_few += 1
            continue

        # -------------------- BEST vs WORST --------------------
        if strategy == "best_worst":
            # Explicit ordering makes ties deterministic.
            ordered = group.sort_values(["average_rank_os", "answer"], ascending=[True, True])
            worst, best = ordered.iloc[0], ordered.iloc[-1]
            delta = float(best["average_rank_os"] - worst["average_rank_os"])

            if delta < min_delta:
                n_skip_delta += 1
                continue
            if jaccard_similarity(best["answer"], worst["answer"]) > similarity_threshold:
                n_skip_sim += 1
                continue

            prompt_data, chosen_data, rejected_data = format_dpo_example(
                question, best["answer"], worst["answer"], conversational_format)
            dpo_list.append({"prompt": prompt_data, "chosen": chosen_data,
                             "rejected": rejected_data, "delta": delta,
                             "raw_question": question})  # helper column for the split

        # ---------------------- PAIRWISE -----------------------
        elif strategy == "pairwise":
            records = group[["answer", "average_rank_os", "model_name"]].to_dict("records")
            prompt_pairs = []
            for r1, r2 in combinations(records, 2):
                delta = abs(r1["average_rank_os"] - r2["average_rank_os"])
                if delta < min_delta:
                    n_skip_delta += 1
                    continue
                chosen_rec, rejected_rec = (r1, r2) if r1["average_rank_os"] > r2["average_rank_os"] else (r2, r1)
                if jaccard_similarity(chosen_rec["answer"], rejected_rec["answer"]) > similarity_threshold:
                    n_skip_sim += 1
                    continue
                prompt_data, chosen_data, rejected_data = format_dpo_example(
                    question, chosen_rec["answer"], rejected_rec["answer"], conversational_format)
                prompt_pairs.append({"prompt": prompt_data, "chosen": chosen_data,
                                     "rejected": rejected_data, "delta": float(delta),
                                     "raw_question": question})

            if max_pairs_per_prompt is not None:   # keep the pairs with the largest gap
                prompt_pairs = sorted(prompt_pairs, key=lambda x: x["delta"], reverse=True)
                prompt_pairs = prompt_pairs[:max_pairs_per_prompt]
            dpo_list.extend(prompt_pairs)

        else:
            raise ValueError("strategy must be 'best_worst' or 'pairwise'")

    print("--- Discard reasons ---")
    print(f"Prompts with fewer than 2 valid answers : {n_skip_few}")
    print(f"Pairs discarded for delta < {min_delta:<5}       : {n_skip_delta}")
    print(f"Pairs discarded for similarity > {similarity_threshold} : {n_skip_sim}\n")

    dpo_df = pd.DataFrame(dpo_list)
    if dpo_df.empty:
        print("WARNING: no triples generated. Consider lowering min_delta or raising similarity_threshold.")
        return pd.DataFrame(), pd.DataFrame()

    print(f"DPO triples generated: {len(dpo_df)}")
    print(f"Mean chosen-rejected gap  : {dpo_df['delta'].mean():.3f}")
    print(f"Median chosen-rejected gap: {dpo_df['delta'].median():.3f}")

    # 1. Full copy (with the `delta` column) for analysis
    path_with_delta = output_parquet_path.replace(".parquet", "_with_delta.parquet")
    dpo_df.drop(columns=["raw_question"]).to_parquet(path_with_delta, index=False, engine="pyarrow")
    print(f"Copy with 'delta' column saved to: {path_with_delta}\n")

    # 2. Deterministic split on UNIQUE PROMPTS: all pairs of a prompt stay on the same side
    unique_questions = pd.Series(dpo_df["raw_question"].unique())
    train_qs = unique_questions.sample(frac=1 - test_size, random_state=random_state)
    test_qs = unique_questions[~unique_questions.isin(train_qs)]
    train_df = dpo_df[dpo_df["raw_question"].isin(train_qs)].copy()
    test_df = dpo_df[dpo_df["raw_question"].isin(test_qs)].copy()
    print(f" -> Training set: {len(train_df)} triples (from {len(train_qs)} prompts)")
    print(f" -> Test set    : {len(test_df)} triples (from {len(test_qs)} prompts)")

    # 3. Final datasets only keep prompt / chosen / rejected
    train_df = train_df.drop(columns=["raw_question", "delta"], errors="ignore")
    test_df = test_df.drop(columns=["raw_question", "delta"], errors="ignore")
    train_path = output_parquet_path.replace(".parquet", "_train.parquet")
    test_path = output_parquet_path.replace(".parquet", "_test.parquet")
    train_df.to_parquet(train_path, index=False, engine="pyarrow")
    test_df.to_parquet(test_path, index=False, engine="pyarrow")
    print(f"Training set saved to: {train_path}")
    print(f"Test set saved to    : {test_path}")
    return train_df, test_df

## 4. Dataset A - Best vs Worst

One pair per prompt: highest-scored vs lowest-scored answer.
Reference stats from the thesis: 313 triples, mean gap 1.746, median gap 1.500
(282 train / 31 test prompts).

In [ ]:
df_train, df_test = build_dpo_dataset(
    df=df,
    output_parquet_path=f"{OUTPUT_DIR}/dpo_best_worst.parquet",
    strategy="best_worst",
    min_delta=0.3,
    similarity_threshold=0.85,
    min_answer_words=3,
    max_answer_words=2000,
    test_size=0.1,
    random_state=42,
)

## 5. Dataset B - Pairwise, delta >= 0.3

All valid pairs per prompt, at most 20 per prompt (largest gaps first).
Reference stats from the thesis: 6097 triples, mean gap 1.272, median gap 1.067.

In [ ]:
df_train, df_test = build_dpo_dataset(
    df=df,
    output_parquet_path=f"{OUTPUT_DIR}/dpo_pairwise.parquet",
    strategy="pairwise",
    min_delta=0.3,
    similarity_threshold=0.85,
    max_pairs_per_prompt=20,
    min_answer_words=3,
    max_answer_words=2000,
    test_size=0.1,
    random_state=42,
)

## 6. Dataset C - Pairwise, delta >= 0.5

Stricter gap threshold and at most 10 pairs per prompt.
Reference stats from the thesis: 3058 triples, mean gap 1.498, median gap 1.300.

In [ ]:
df_train, df_test = build_dpo_dataset(
    df=df,
    output_parquet_path=f"{OUTPUT_DIR}/dpo_pairwise_strict.parquet",
    strategy="pairwise",
    min_delta=0.5,
    similarity_threshold=0.85,
    max_pairs_per_prompt=10,
    min_answer_words=3,
    max_answer_words=2000,
    test_size=0.1,
    random_state=42,
)

## 7. Sanity check - score-gap statistics of a saved dataset

In [ ]:
import pandas as pd

PARQUET_FILE = f"{OUTPUT_DIR}/dpo_pairwise_strict_with_delta.parquet"
stats_df = pd.read_parquet(PARQUET_FILE)
assert "delta" in stats_df.columns, "The 'delta' column is missing from the dataset."

print("=" * 50)
print(f"File           : {PARQUET_FILE}")
print(f"Number of pairs: {len(stats_df)}")
print(f"Mean gap       : {stats_df['delta'].mean():.4f}")
print(f"Median gap     : {stats_df['delta'].median():.4f}")
print("=" * 50)